# Bedrock: walkthrough
Reruns every result in the dashboard and the research note.

Before running:
```
pip install -r requirements.txt && pip install -e .
python -m bedrock.download
python -m bedrock.pipeline
```

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from bedrock import depth, speculation, selloff, stress
from bedrock.config import PROCESSED
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
r = lambda n: pd.read_parquet(PROCESSED / f'{n}.parquet')
stocks, indices, sectors, share = r('stocks'), r('indices'), r('sectors'), r('shareholding')
flows, macro = r('foreign_flows'), r('macro')
for n, d in dict(stocks=stocks, flows=flows, macro=macro).items():
    c = 'date' if 'date' in d else 'period'
    print(f'{n:<7} {len(d):>8,} rows  {d[c].min():%Y-%m} -> {d[c].max():%Y-%m}')

stocks   220,193 rows  2013-01 -> 2026-11
flows        141 rows  2015-01 -> 2026-09
macro        263 rows  2005-01 -> 2026-11


## Module A: Market Depth
The headline numbers use ordinary shares only. Tiers come from days to exit; K-means is the cross-check.

In [2]:
snap = depth.depth_snapshot(stocks, share, sectors)
depth.headline(snap)

{'n_stocks': 64,
 'as_of': '2026-11-16',
 'pct_over_5d': 0.796875,
 'pct_over_20d': 0.671875,
 'pct_no_trade_in_window': 0.109375,
 'top5_share_of_value': 0.7409041291608479}

In [3]:
snap[['code','name','tier','days_to_exit','adv_kes','zero_trade_share','amihud','foreign_pct']].head(15)

,code,name,tier,days_to_exit,adv_kes,zero_trade_share,amihud,foreign_pct
0,SCOM,Safaricom Plc,1 Deep,0.218852,2.284646e+08,0.000000,0.000096,0.0613
1,EQTY,Equity Group Holdings Plc,1 Deep,0.253718,1.970691e+08,0.008333,0.000156,0.4151
2,KCB,KCB Group Plc,1 Deep,0.398475,1.254783e+08,0.000000,0.000153,0.0842
3,DTK,Diamond Trust Bank Kenya Ltd,1 Deep,1.578926,3.166709e+07,0.000000,0.000984,0.5110
4,COOP,Co-operative Bank of Kenya Ltd,1 Deep,1.675179,2.984755e+07,0.000000,0.000648,0.0025
5,KPLC,Kenya Power and Lighting Company Plc,2 Moderate,2.269110,2.203507e+07,0.000000,0.000789,0.0636
6,IMH,I & M Holdings Plc,2 Moderate,2.473435,2.021480e+07,0.000000,0.001731,0.1778
7,NCBA,NCBA Group Plc,2 Moderate,2.614976,1.912063e+07,0.066667,0.001512,0.0038
8,EABL,East African Breweries Ltd,2 Moderate,2.671492,1.871613e+07,0.000000,0.002054,0.0966
9,ABSA,ABSA Bank Kenya Plc,2 Moderate,2.880669,1.735708e+07,0.000000,0.000968,0.7234


In [4]:
pd.crosstab(snap['tier'], snap['cluster'], margins=True)  # rule-based tiers vs K-means clusters

cluster,1.0,2.0,3.0,All
tier,,,,
1 Deep,5,0,0,5
2 Moderate,16,0,0,16
3 Thin,5,18,0,23
4 Frozen,0,13,3,16
All,26,31,3,60


In [5]:
hist = r('outputs/depth_history') if (PROCESSED/'outputs/depth_history.parquet').exists() else None
hist[['as_of','n_stocks','pct_over_5d','pct_over_20d','top5_share_of_value']].iloc[::4]

,as_of,n_stocks,pct_over_5d,pct_over_20d,top5_share_of_value
0,2013-09-30,56,0.892857,0.535714,0.747615
4,2014-09-30,57,0.807018,0.526316,0.731112
8,2015-09-30,59,0.881356,0.610169,0.818174
12,2016-09-30,61,0.918033,0.754098,0.882928
16,2017-09-29,61,0.918033,0.704918,0.860039
20,2018-09-28,61,0.901639,0.737705,0.883766
24,2019-09-30,62,0.919355,0.838710,0.923983
28,2020-09-30,60,0.933333,0.866667,0.938900
32,2021-09-30,60,0.933333,0.866667,0.924707
36,2022-09-30,62,0.935484,0.887097,0.939333


## Module C: Speculation Watch

In [6]:
feats = speculation.build_features(stocks, indices)
scored = speculation.detect(feats)
print(f"flagged {scored['flag'].sum():,} of {len(scored):,} stock-days; watch list {scored['watch'].sum():,}")
speculation.validation(scored, stocks.groupby('code')['value_kes'].mean())

flagged 1,659 of 165,898 stock-days; watch list 1,070


,group,size,direction,fwd_5d_mean,fwd_5d_count,fwd_20d_mean,fwd_20d_count
0,other big moves,large,down,0.016966,2991,0.024313,2991
1,other big moves,large,up,0.001342,3501,0.002439,3501
2,other big moves,small,down,0.022045,5260,0.031727,5260
3,other big moves,small,up,0.001669,6098,0.008016,6098
4,watch list,large,down,0.024660,157,0.048086,157
5,watch list,large,up,0.014024,234,-0.027351,234
6,watch list,small,down,0.028015,351,0.029956,351
7,watch list,small,up,0.044969,240,0.113579,240


## Module B: Sell-off Alert
This is a walk-forward test: every month the model is retrained only on data that existed at that time.

In [7]:
sb = selloff.run(flows, macro, stocks, indices)
sb['evaluation']

,model,brier,brier_skill_vs_base,auc,n_high_risk_calls,hit_rate_high_risk,base_rate,n_periods
0,climatology,0.189829,0.000000,0.635345,0,NaN,0.25641,78
1,logistic,0.195766,-0.031275,0.656034,10,0.400000,0.25641,78
2,xgboost,0.173284,0.087159,0.639655,17,0.588235,0.25641,78


In [8]:
live = sb['live']
print(f"{live['period']:%b %Y}: P(heavy selling next period) = {live['probability']:.0%} -> {live['signal']} ({live['model']})")
print('dropped (stale):', live['features_dropped'])
live['drivers'].head(8)

Sep 2026: P(heavy selling next period) = 53% -> HIGH RISK (xgboost)
dropped (stale): []


,feature,value,contribution,label,direction
0,flow,-3611.0,0.674625,Net foreign flow this period,raises risk
1,flow_3,-5668.0,0.510156,"Net foreign flow, last 3 periods",raises risk


## Stress test: foreigners sell KES 2bn over 5 days

In [9]:
res = stress.run_scenario(snap)
stress.summarize(res, 2e9)

{'total_sell_kes': 2000000000.0,
 'n_hit': 46,
 'n_trapped': 16,
 'n_no_market': 4,
 'value_trapped_kes': 1300649917.5982113,
 'weighted_impact': -0.03995076299157914,
 'worst': [{'code': 'PORT',
   'price_impact': -0.25104375190050476,
   'days_to_absorb': 354.3299818983405},
  {'code': 'CRWN',
   'price_impact': -0.20508019898274826,
   'days_to_absorb': 391.8842441147821},
  {'code': 'NBV',
   'price_impact': -0.18390908369253842,
   'days_to_absorb': 276.3137120033473},
  {'code': 'KUKZ',
   'price_impact': -0.1741827870848198,
   'days_to_absorb': 391.3069135202459},
  {'code': 'BRIT',
   'price_impact': -0.15575179159116712,
   'days_to_absorb': 143.05213831437663}]}

In [10]:
res.head(15)

,code,name,sector,tier,foreign_pct,foreign_value_kes,sell_kes,adv_kes,sell_vs_adv,days_to_absorb,price_impact,status,speculative_recently
0,PORT,East African Portland Cement Ltd,Construction and Allied,3 Thin,0.2935,2.859424e+09,1.562813e+07,2.205307e+05,14.173199,354.329982,-0.251044,trapped,False
1,CRWN,Crown Paints Kenya Plc,Construction and Allied,3 Thin,0.4411,3.846248e+09,2.102159e+07,2.682118e+05,15.675370,391.884244,-0.205080,trapped,False
2,NBV,Nairobi Business Ventures Ltd,Commercial and Services,4 Frozen,0.3212,5.609078e+08,3.065631e+06,5.547374e+04,11.052548,276.313712,-0.183909,trapped,False
3,KUKZ,Kakuzi Plc,Agricultural,4 Frozen,0.3030,2.559623e+09,1.398957e+07,1.787544e+05,15.652277,391.306914,-0.174183,trapped,False
4,BRIT,Britam Holdings Plc,Insurance,2 Moderate,0.4259,1.923808e+10,1.051454e+08,3.675071e+06,5.722086,143.052138,-0.155752,trapped,False
5,EGAD,Eaagads Ltd,Agricultural,4 Frozen,0.2130,1.839075e+08,1.005143e+06,1.175228e+05,1.710550,42.763747,-0.101896,trapped,False
6,KAPC,Kapchorua Tea Kenya Plc,Agricultural,3 Thin,0.2896,1.528303e+09,8.352909e+06,8.196633e+05,2.038132,50.953291,-0.068406,trapped,False
7,NMG,Nation Media Group Plc,Commercial and Services,3 Thin,0.1098,3.188099e+08,1.742450e+06,3.864408e+05,0.901794,22.544842,-0.063391,trapped,False
8,IMH,I & M Holdings Plc,Banking,2 Moderate,0.1778,2.575702e+10,1.407745e+08,2.021480e+07,1.392786,34.819650,-0.051599,trapped,False
9,CARB,Carbacid Investments Plc,Manufacturing and Allied,3 Thin,0.1041,1.116917e+09,6.104488e+06,1.431704e+06,0.852759,21.318963,-0.048471,trapped,False
